In [1]:
# %% [markdown]
# # 01 - Data Preparation
# **Module:** Deep Learning (MOD006565)
# **Description:** Load Food-101, filter to 50 classes, split into train/val/test and save indices to disk.

# %% [markdown]
# ## Imports

# %%
import os
import json
import numpy as np
import tensorflow_datasets as tfds

# %% [markdown]
# ## Configuration

# %%
DATASET_DIR  = '/workspace/food101-dl-project/dataset'
OUTPUT_DIR   = '/workspace/food101-dl-project/outputs'
SPLIT_DIR    = '/workspace/food101-dl-project/splits'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

SELECTED_CLASSES = [
    'apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare',
    'bibimbap', 'bread_pudding', 'buffalo_wings', 'caprese_salad', 'carrot_cake',
    'cheesecake', 'chicken_curry', 'chicken_wings', 'chocolate_cake', 'churros',
    'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'falafel',
    'french_fries', 'fried_rice', 'grilled_salmon', 'guacamole', 'hamburger',
    'hot_and_sour_soup', 'hotdog', 'ice_cream', 'lasagna', 'nachos',
    'omelette', 'onion_rings', 'pancakes', 'pizza', 'ramen',
    'ravioli', 'red_velvet_cake', 'risotto', 'samosa', 'sashimi',
    'shrimp_and_grits', 'spaghetti_bolognese', 'steak', 'sushi', 'tacos',
    'tiramisu', 'waffles', 'spring_rolls', 'pho', 'creme_brulee'
]

NUM_CLASSES = len(SELECTED_CLASSES)
print(f"Number of selected classes : {NUM_CLASSES}")

# %% [markdown]
# ## Load Dataset

# %%
print("Loading Food-101 from disk...")

ds, info = tfds.load(
    'food101',
    data_dir=DATASET_DIR,
    with_info=True,
    as_supervised=True,
    download=False
)

all_classes = info.features['label'].names
print(f"Total classes in dataset   : {len(all_classes)}")
print(f"Total train examples       : {info.splits['train'].num_examples}")
print(f"Total val examples         : {info.splits['validation'].num_examples}")

# %% [markdown]
# ## Build Class Mappings

# %%
class_to_orig_idx = {cls: idx for idx, cls in enumerate(all_classes)}
selected_orig_idx = [class_to_orig_idx[cls] for cls in SELECTED_CLASSES if cls in class_to_orig_idx]
orig_to_new_idx   = {orig: new for new, orig in enumerate(selected_orig_idx)}
selected_set      = set(selected_orig_idx)

print(f"Classes matched in dataset : {len(selected_orig_idx)}")

# Save class mapping for use in other notebooks
mapping = {
    'selected_classes': SELECTED_CLASSES,
    'orig_to_new_idx': {str(k): v for k, v in orig_to_new_idx.items()}
}
with open(f'{SPLIT_DIR}/class_mapping.json', 'w') as f:
    json.dump(mapping, f, indent=4)

print(f"Class mapping saved to     : {SPLIT_DIR}/class_mapping.json")

# %% [markdown]
# ## Filter and Split Data
#
# Train split comes from the official Food-101 train set.
# Validation set is split 50/50 from the official validation set to produce
# a clean validation and held-out test set. This prevents data leakage as
# the split is performed before any preprocessing or augmentation.

# %%
def extract_split(tfds_split, selected_set, orig_to_new_idx):
    images, labels = [], []
    for image, label in tfds_split:
        orig_label = int(label.numpy())
        if orig_label in selected_set:
            images.append(image.numpy())
            labels.append(orig_to_new_idx[orig_label])
    return images, labels

print("Extracting train split...")
train_images, train_labels = extract_split(ds['train'], selected_set, orig_to_new_idx)
print(f"Train examples extracted   : {len(train_images)}")

print("Extracting validation split...")
val_images, val_labels = extract_split(ds['validation'], selected_set, orig_to_new_idx)
print(f"Validation examples        : {len(val_images)}")

# Split validation 50/50 into val and test
split_point  = len(val_images) // 2
indices      = np.random.RandomState(42).permutation(len(val_images))

val_idx  = indices[:split_point]
test_idx = indices[split_point:]

val_images_final  = [val_images[i] for i in val_idx]
val_labels_final  = [val_labels[i] for i in val_idx]
test_images_final = [val_images[i] for i in test_idx]
test_labels_final = [val_labels[i] for i in test_idx]

print(f"Final validation examples  : {len(val_images_final)}")
print(f"Final test examples        : {len(test_images_final)}")

# %% [markdown]
# ## Save Splits to Disk

# %%
print("Saving splits to disk...")

np.save(f'{SPLIT_DIR}/train_images.npy', np.array(train_images, dtype=np.uint8))
np.save(f'{SPLIT_DIR}/train_labels.npy', np.array(train_labels, dtype=np.int64))
np.save(f'{SPLIT_DIR}/val_images.npy',   np.array(val_images_final, dtype=np.uint8))
np.save(f'{SPLIT_DIR}/val_labels.npy',   np.array(val_labels_final, dtype=np.int64))
np.save(f'{SPLIT_DIR}/test_images.npy',  np.array(test_images_final, dtype=np.uint8))
np.save(f'{SPLIT_DIR}/test_labels.npy',  np.array(test_labels_final, dtype=np.int64))

print("Splits saved successfully.")
print(f"Location : {SPLIT_DIR}")
print(f"Files    : train_images.npy, train_labels.npy")
print(f"           val_images.npy,   val_labels.npy")
print(f"           test_images.npy,  test_labels.npy")

# %% [markdown]
# ## Summary

# %%
print("Data Preparation Summary")
print(f"Selected classes : {NUM_CLASSES}")
print(f"Train set        : {len(train_images)}")
print(f"Validation set   : {len(val_images_final)}")
print(f"Test set         : {len(test_images_final)}")
print(f"Total            : {len(train_images) + len(val_images_final) + len(test_images_final)}")
print("No preprocessing or augmentation applied at this stage.")
print("Split is fixed with random seed 42 for reproducibility.")

Number of selected classes : 50
Loading Food-101 from disk...


2026-04-16 16:18:52.716163: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 16:18:53.555722: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-04-16 16:18:55.598450: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Total classes in dataset   : 101
Total train examples       : 75750
Total val examples         : 25250
Classes matched in dataset : 48
Class mapping saved to     : /workspace/food101-dl-project/splits/class_mapping.json
Extracting train split...


2026-04-16 16:19:26.761495: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Train examples extracted   : 36000
Extracting validation split...


2026-04-16 16:19:36.732313: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Validation examples        : 12000
Final validation examples  : 6000
Final test examples        : 6000
Saving splits to disk...


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (36000,) + inhomogeneous part.